In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

def amend_bdate(cust_az_existing, random_state=42):
    np.random.seed(random_state)
    
    df = cust_az_existing.copy()
    
    n = len(df)
    
    # --- Create realistic age distribution ---
    # Younger group (commuters / casual riders)
    group1 = np.random.normal(loc=32, scale=8, size=int(n * 0.6))
    
    # Mid-age group (enthusiasts / higher spenders)
    group2 = np.random.normal(loc=48, scale=10, size=int(n * 0.4))
    
    ages = np.concatenate([group1, group2])
    
    # Ensure correct length
    if len(ages) < n:
        ages = np.append(ages, np.random.normal(40, 10, n - len(ages)))
    
    # Clean ages
    ages = np.clip(ages, 18, 75).astype(int)
    
    # Shuffle so distribution isn't ordered
    np.random.shuffle(ages)
    
    # --- Convert age to birthdate ---
    today = pd.Timestamp.today()
    
    birthdates = [
        today - pd.DateOffset(years=int(age)) - pd.DateOffset(days=np.random.randint(0, 365))
        for age in ages
    ]
    
    # Assign back
    df['BDATE'] = [d.date() for d in pd.to_datetime(birthdates)]
    
    return df

# --------------------------------------------------
# CUSTOMER GENERATION - 100 NEW CUSTOMERS
# --------------------------------------------------

def generate_customers(df_existing, n_new=100, random_state=42):
    """
    Generate new customers based on existing dataset patterns.
    """
    np.random.seed(random_state)

    max_id = df_existing['cst_id'].max()
    new_rows = []

    for i in range(1, n_new + 1):
        sample = df_existing.sample(1).iloc[0]

        new_id = max_id + i
        new_key = f"AW{new_id:08d}"

        new_rows.append({
            'cst_id': new_id,
            'cst_key': new_key,
            'cst_firstname': sample['cst_firstname'],
            'cst_lastname': sample['cst_lastname'],
            'cst_marital_status': sample['cst_marital_status'],
            'cst_gndr': sample['cst_gndr'],
            'cst_create_date': sample['cst_create_date']
        })

    return pd.DataFrame(new_rows)


# --------------------------------------------------
# ADD NEW CUSTOMERS TO CUST_AZ12
# --------------------------------------------------

def update_cust_az(cust_new, cust_az_amended, n_records=100, random_state=42):
    """
    Create CUST_AZ records for new customers and append to existing dataset.
    """
    np.random.seed(random_state)

    df_new = cust_new[['cst_key', 'cst_gndr']].copy().head(n_records)

    df_new['CID'] = 'NAS' + df_new['cst_key'].astype(str)

    df_new['GEN'] = df_new['cst_gndr'].map({
        'M': 'Male',
        'F': 'Female'
    })

    # More realistic DOB generation
    today = datetime.today()
    ages = np.random.randint(18, 76, size=len(df_new))

    birthdates = []
    for age in ages:
        # Random day within that year range
        days_offset = np.random.randint(0, 365)
        dob = today - timedelta(days=int(age * 365 + days_offset))
        birthdates.append(dob)

    df_new['BDATE'] = pd.Series(birthdates).dt.strftime('%d/%m/%Y')
    df_new['BDATE'] = pd.to_datetime(df_new['BDATE'], dayfirst=True).dt.strftime('%Y-%m-%d')

    df_new_final = df_new[['CID', 'BDATE', 'GEN']]

    final_df = pd.concat([cust_az_amended, df_new_final], ignore_index=True)

    return final_df


# --------------------------------------------------
# EXPAND EXISTING SALES DATA
# --------------------------------------------------

def expand_sales_data(df, target_rows=600000, random_state=42):
    """
    Expand sales dataset to target size using realistic distributions.
    """
    np.random.seed(random_state)

    df = df.copy()

    if len(df) >= target_rows:
        print("Dataset already meets or exceeds target rows.")
        return df

    rows_to_generate = target_rows - len(df)

    product_keys = df['sls_prd_key'].dropna().unique()
    customer_ids = df['sls_cust_id'].dropna().unique()
    product_base_prices = df.groupby('sls_prd_key')['sls_price'].median()

    base_year = 2015

    max_order = (
        df['sls_ord_num']
        .astype(str)
        .str.replace('SO', '', regex=False)
        .astype(int)
        .max()
    )

    # Generate dates
    order_dates_raw = np.random.choice(df['sls_order_dt'], size=rows_to_generate)
    order_dates = pd.to_datetime(order_dates_raw, format='%Y%m%d', errors='coerce')

    order_dates += pd.to_timedelta(
        np.random.randint(1, 3000, size=rows_to_generate), unit='D'
    )

    ship_dates = order_dates + pd.to_timedelta(np.random.randint(1, 10, size=rows_to_generate), unit='D')
    due_dates = ship_dates + pd.to_timedelta(np.random.randint(1, 5, size=rows_to_generate), unit='D')

    # Products & customers
    products = np.random.choice(product_keys, size=rows_to_generate)
    customers = np.random.choice(customer_ids, size=rows_to_generate)

    # Pricing
    base_prices = pd.Series(products).map(product_base_prices).fillna(100).values
    year_diff = order_dates.year - base_year
    prices = np.round(base_prices * (1.02 ** year_diff), 2)

    # Quantities
    quantities = np.random.choice(
        [1,2,3,4,5],
        size=rows_to_generate,
        p=[0.45,0.30,0.15,0.07,0.03]
    )

    sales = np.round(prices * quantities, 2)

    # Order numbers
    order_numbers = [f"SO{max_order + i + 1}" for i in range(rows_to_generate)]

    new_data = pd.DataFrame({
        'sls_ord_num': order_numbers,
        'sls_prd_key': products,
        'sls_cust_id': customers,
        'sls_order_dt': order_dates.strftime('%Y%m%d'),
        'sls_ship_dt': ship_dates.strftime('%Y%m%d'),
        'sls_due_dt': due_dates.strftime('%Y%m%d'),
        'sls_sales': sales,
        'sls_quantity': quantities,
        'sls_price': prices
    })

    final_df = pd.concat([df, new_data], ignore_index=True)

    final_df['order_year'] = pd.to_datetime(
        final_df['sls_order_dt'], format='%Y%m%d', errors='coerce'
    ).dt.year

    return final_df


# --------------------------------------------------
# GENERATE SALES FOR NEW CUSTOMERS
# --------------------------------------------------

def generate_sales_for_customers(cust_df, sales_df, random_state=42):
    """
    Generate realistic sales data for new customers.
    """
    np.random.seed(random_state)

    sales_df = sales_df.copy()

    sales_df['sls_order_dt'] = pd.to_datetime(
        sales_df['sls_order_dt'],
        format='%Y%m%d',
        errors='coerce'
    )

    max_date = sales_df['sls_order_dt'].max()
    cutoff_date = max_date - pd.DateOffset(months=12)

    recent_sales = sales_df[sales_df['sls_order_dt'] >= cutoff_date]

    product_price = recent_sales.groupby('sls_prd_key')['sls_price'].median()
    product_keys = product_price.index.values

    max_order = (
        sales_df['sls_ord_num']
        .astype(str)
        .str.replace('SO', '', regex=False)
        .astype(int)
        .max()
    )

    qty_choices = [1,2,3,4,5,6,7]
    qty_probs   = [0.45,0.30,0.10,0.07,0.04,0.02,0.02]

    order_choices = [1,2,3,4,5]
    order_probs   = [0.35,0.30,0.20,0.10,0.05]

    rows = []
    order_counter = 1

    for _, cust in cust_df.iterrows():

        num_orders = np.random.choice(order_choices, p=order_probs)

        for _ in range(num_orders):

            product = np.random.choice(product_keys)
            base_price = product_price.get(product, 100)

            order_date = cutoff_date + pd.to_timedelta(
                np.random.randint(0, (max_date - cutoff_date).days),
                unit='D'
            )

            ship_date = order_date + pd.to_timedelta(np.random.randint(1,10), unit='D')
            due_date  = ship_date + pd.to_timedelta(np.random.randint(1,5), unit='D')

            qty = np.random.choice(qty_choices, p=qty_probs)
            price = round(base_price * np.random.uniform(0.95,1.05), 2)
            sales_val = round(qty * price, 2)

            order_num = f"SO{max_order + order_counter}"
            order_counter += 1

            rows.append({
                'sls_ord_num': order_num,
                'sls_prd_key': product,
                'sls_cust_id': cust['cst_id'],
                'sls_order_dt': order_date.strftime('%Y%m%d'),
                'sls_ship_dt': ship_date.strftime('%Y%m%d'),
                'sls_due_dt': due_date.strftime('%Y%m%d'),
                'sls_sales': sales_val,
                'sls_quantity': qty,
                'sls_price': price,
                'order_year': order_date.year
            })
   

    return pd.DataFrame(rows)

# --------------------------------------------------
# GENERATE COUNTRY DATA FOR NEW CUSTOMERS
# --------------------------------------------------

def update_loc_a101(cust_new, loc_existing, n_records=200, random_state=42):
    """
    Create LOC_A101 records for new customers and append to existing dataset.
    Ensures CID is in the format AW-XXXXXXXX (8 digits after AW-).
    """
    import random
    import pandas as pd

    random.seed(random_state)

    # Take the first n_records from the new customers
    df_new = cust_new[['cst_id']].copy().head(n_records)

    # Format CID as AW-XXXXXXXX with leading zeros (8 digits after AW-)
    df_new['CID'] = df_new['cst_id'].apply(lambda x: f"AW-{str(x).zfill(8)}")

    # Assign countries randomly
    countries = [
        "United States",
        "Australia",
        "United Kingdom",
        "Germany",
        "France",
        "Canada"
    ]
    df_new['CNTRY'] = [random.choice(countries) for _ in range(len(df_new))]

    # Keep only CID and CNTRY columns
    df_new_final = df_new[['CID', 'CNTRY']]

    # Append to existing LOC_A101 data
    updated_LOC_A101 = pd.concat([loc_existing, df_new_final], ignore_index=True)

    return updated_LOC_A101